# Ejercicios: dovetailing de reconocedores

Este cuaderno ilustra la idea de ejecutar reconocedores en paralelo.

Si un lenguaje y su complemento son reconocibles, podemos decidir el lenguaje alternando pasos de ambos reconocedores.

In [ ]:
def reconocedor_pares(n):
    paso = 0
    while True:
        paso += 1
        if n % 2 == 0 and paso >= 2:
            yield "acepta"
        else:
            yield "sigue"


def reconocedor_impares(n):
    paso = 0
    while True:
        paso += 1
        if n % 2 == 1 and paso >= 3:
            yield "acepta"
        else:
            yield "sigue"


def decidir_por_dovetailing(n, limite=20):
    r_l = reconocedor_pares(n)
    r_c = reconocedor_impares(n)
    for paso in range(1, limite + 1):
        if next(r_l) == "acepta":
            return "par", paso
        if next(r_c) == "acepta":
            return "impar", paso
    return "sin respuesta", limite

## Probar el decisor

El ejemplo usa paridad porque es decidible, pero la estructura imita el argumento general.

In [ ]:
for n in range(6):
    resultado, paso = decidir_por_dovetailing(n)
    print(f"n={n}: {resultado} en ronda {paso}")

## Reconocedor sin complemento

Si solo tenemos un reconocedor, podemos confirmar casos positivos, pero quizá no terminar en casos negativos.

In [ ]:
def intentar_reconocer_par(n, limite=5):
    r = reconocedor_pares(n)
    for paso in range(1, limite + 1):
        if next(r) == "acepta":
            return "acepta", paso
    return "no concluye dentro del límite", limite


for n in [2, 3]:
    print(n, intentar_reconocer_par(n))

## Para experimentar

1. Cambia los pasos mínimos antes de aceptar.
2. Crea reconocedores para múltiplos de 3 y no múltiplos de 3.
3. Explica por qué este esquema no resuelve el problema de la parada.

In [ ]:
# ── Verificación automática ──────────────────────────────────────────────────

def dovetailing_simple(reconocedores, max_pasos=1000):
    """
    Simula dovetailing: ejecuta cada reconocedor de forma intercalada.
    Cada reconocedor es un iterador que yield True cuando acepta.
    """
    activos = list(reconocedores)
    for paso in range(max_pasos):
        nuevos = []
        for r in activos:
            try:
                resultado = next(r)
                if resultado:
                    return True
            except StopIteration:
                pass
            else:
                nuevos.append(r)
        activos = nuevos
    return False

def reconocedor_par():
    for n in range(0, 100, 2):
        yield n == 4  # "acepta" solo el 4

def reconocedor_impar():
    for n in range(1, 100, 2):
        yield n == 7  # "acepta" solo el 7

assert dovetailing_simple([reconocedor_par(), reconocedor_impar()]),     "Dovetailing debe encontrar al menos un reconocedor que acepte"
print("✓ Dovetailing funciona correctamente")